### Import Libraries


In [2]:
from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait as WOW
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager


In [3]:
from selenium.webdriver.support.ui import WebDriverWait as WDW #used this code to import wait time feauture and shortened as WDW 

### Setup and Configure Selenium WebDriver

In [4]:
print("Setting up Webddriver....")
chrome_opt = Options() # Intialize the chrome webdriver
chrome_opt.add_argument("--headless")
chrome_opt.add_argument("--disable-gpu")
chrome_opt.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/131.0.6778.265 Safari/537.36")
print("configuration done!")

Setting up Webddriver....
configuration done!


In [5]:
# Setting Up Webdriver: Installation and Initialization
print("Installing Chrome Webdriver")
service = Service(ChromeDriverManager().install())
print("Final Setup")
driver = webdriver.Chrome(service=service, options=chrome_opt)
print("done!!")


Installing Chrome Webdriver
Final Setup
done!!


### Making a connection to the webpage


In [6]:
URL = "https://www.glassesusa.com/eyeglasses-collection"

In [10]:
print(f"Visiting {URL} page")
driver.get(URL)

# Further Instructions
try:
    print("Waiting for product tiles to load")
    WDW(driver, 40).until(EC.presence_of_element_located((By.CLASS_NAME, 'VFCkJ'))) #Here we are using a class name from the body tag to ensure we are targeting the glasses lists.
    print("Done, Proceed!")
except TimeoutError as e:
    print(f"Expected tag did not load on time: {e}")

Visiting https://www.glassesusa.com/eyeglasses-collection page
Waiting for product tiles to load


TimeoutException: Message: 
Stacktrace:
	chromedriver!GetHandleVerifier [0x43cdf3+10b03]
	chromedriver!GetHandleVerifier [0x43cf24+10c34]
	chromedriver!(No symbol) [0x222120]
	chromedriver!(No symbol) [0x26abca]
	chromedriver!(No symbol) [0x26ae6b]
	chromedriver!(No symbol) [0x2ad0b2]
	chromedriver!(No symbol) [0x28db54]
	chromedriver!(No symbol) [0x2aa9a5]
	chromedriver!(No symbol) [0x28d8a6]
	chromedriver!(No symbol) [0x260229]
	chromedriver!(No symbol) [0x260fe4]
	chromedriver!GetHandleVerifier [0x6a48b9+2785c9]
	chromedriver!GetHandleVerifier [0x69feb5+273bc5]
	chromedriver!GetHandleVerifier [0x6be06b+291d7b]
	chromedriver!GetHandleVerifier [0x455cc8+299d8]
	chromedriver!GetHandleVerifier [0x45d9fd+3170d]
	chromedriver!GetHandleVerifier [0x4458c8+195d8]
	chromedriver!GetHandleVerifier [0x445a92+197a2]
	chromedriver!GetHandleVerifier [0x42ee9a+2baa]
	KERNEL32!BaseThreadInitThunk [0x76f6fcc9+19]
	ntdll!RtlGetAppContainerNamedObjectPath [0x776282ae+11e]
	ntdll!RtlGetAppContainerNamedO

In [31]:
content = driver.page_source
page = BeautifulSoup(content, "html.parser")

### Data Extraction

In [ ]:
product_tiles = page.find_all("div", class_="jsqV2")
print(f"Found {len(product_tiles)} products")

Found 25 products


In [ ]:
products = []

for tile in product_tiles:
    product_info = tile.find('div', class_='NUGDI')
#prod-image-holder
    if product_info:
        name_tag = product_info.find('div', class_='CBjRE')
        name = name_tag.text if name_tag else "Unknown"

        # brand
        brand_tag = product_info.find('div', class_='catalog-name')
        brand = brand_tag.text if brand_tag else "Unknown"
        
        #new price logic
        


        # price
        price_container = product_info.find('div', class_='prod-bot')
        if price_container:
            # former price
            former_price_tag = price_container.select_one('[data-test-name="regularPrice"]')
            former_price = former_price_tag.text if former_price_tag else 'Unknown'
            # Current price
            current_price_tag = price_container.select_one('a[data-test-name="productTitle"]')
            current_price = current_price_tag.text if current_price_tag else 'Unknown'
        else:
            former_price = current_price = "Unknown"
    else:
        brand = name = former_price = current_price = "Unknown"

    data = {
        "Product_Name": name,
        "Brand": brand,
        "Former_Price": former_price,
        "Current_Price": current_price
    }
    print(data)
    products.append(data)

NameError: name 'product_tiles' is not defined

In [12]:
# Cell 9
# Step 3 - Data Storage: store the extracted data in CSV and JSON formats
import csv
import json
# Save to CSV file
column_name = products[0].keys() # get the column names
with open('glassesdotcom.csv', mode='w', newline='', encoding='utf-8') as csv_file: # open up the file with context manager
    dict_writer = csv.DictWriter(csv_file, fieldnames=column_name)
    dict_writer.writeheader()
    dict_writer.writerows(products)
print(f"Saved {len(products)} records to CSV in the extracted data folder.")

# Save to JSON file
with open("glassesdotcom.json", mode='w') as json_file:
    json.dump(products, json_file, indent=4)
print(f"Saved {len(products)} records to JSON in the extracted data folder.")

# close the browser
driver.quit()
print("End of Web Extraction")

Saved 26 records to CSV in the extracted data folder.
Saved 26 records to JSON in the extracted data folder.
End of Web Extraction
